# Conversion Driver Analysis & Efficiency Optimization

## Bank Term Deposit Campaign Targeting

### Business question
Which customer segments should the bank prioritize in telemarketing campaigns to maximize term deposit conversions (Certificate of Deposit), and which segments or calling behaviors are likely wasting call volume?

A term deposit is a fixed-term, fixed-rate savings product

#### Purpose of this notebook
Full EDA behind the targeting recommendation: customer-profile segment conversion, prior contact history, call efficiency (touches per lead), and economic-condition sensitivity.

### Baseline Conversion Rate
What fraction converted?

In [0]:
USE workspace.bank_marketing;

SELECT
  COUNT(*) AS total_contacts,
  SUM(CAST(converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(converted AS INT)), 4) AS overall_conversion_rate
FROM fact_contacts;

total_contacts,total_conversions,overall_conversion_rate
41176,4639,0.1127


### Observation
Out of 41,176 contacts, about 11.3% converted (roughly 1 in 9 calls). That's the baseline — any segment converting meaningfully above ~11% is a good targeting candidate; anything well below it is a candidate to deprioritize.

In [0]:
CREATE OR REPLACE VIEW v_segment_conversion AS
WITH base AS (
  SELECT
    f.contact_key,
    f.converted,
    c.job,
    c.education,
    c.marital_status,
    c.age_bracket
  FROM fact_contacts f
  JOIN dim_customer_profile c ON f.customer_profile_key = c.customer_profile_key
)
SELECT 'job' AS segment_dimension, job AS segment_value,
       COUNT(*) AS contacts, SUM(CAST(converted AS INT)) AS conversions,
       ROUND(AVG(CAST(converted AS INT)), 4) AS conversion_rate
FROM base GROUP BY job

UNION ALL

SELECT 'education', education,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4)
FROM base GROUP BY education

UNION ALL

SELECT 'marital_status', marital_status,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4)
FROM base GROUP BY marital_status

UNION ALL

SELECT 'age_bracket', age_bracket,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4)
FROM base GROUP BY age_bracket;

In [0]:
SELECT segment_value AS job, contacts, conversions, conversion_rate
FROM v_segment_conversion
WHERE segment_dimension = 'job'
ORDER BY conversion_rate DESC;

job,contacts,conversions,conversion_rate
student,875,275,0.3143
retired,1718,434,0.2526
unemployed,1014,144,0.142
admin.,10419,1351,0.1297
management,2924,328,0.1122
unknown,330,37,0.1121
technician,6739,730,0.1083
self-employed,1421,149,0.1049
housemaid,1060,106,0.1
entrepreneur,1456,124,0.0852


### Observations
Job is a strong signal. **Students (31%) and retirees (25%)** convert at roughly 2.5–3x the baseline rate, and both have a decent sample size (875 and 1,718 contacts) — not a fluke. At the other end, **blue-collar (7%) and services (8%)** — two of the largest segments by volume (9,253 and 3,967 contacts) — convert well below baseline. That combination (large volume + low conversion) is exactly where the most wasted call effort is likely concentrated.

In [0]:
SELECT segment_value AS education, contacts, conversions, conversion_rate
FROM v_segment_conversion
WHERE segment_dimension = 'education'
ORDER BY conversion_rate DESC;

education,contacts,conversions,conversion_rate
illiterate,18,4,0.2222
unknown,1730,251,0.1451
university.degree,12164,1669,0.1372
professional.course,5240,595,0.1135
high.school,9512,1031,0.1084
basic.4y,4176,428,0.1025
basic.6y,2291,188,0.0821
basic.9y,6045,473,0.0782


### Observations
illiterate shows the single highest rate (22%), but on only 18 contacts total — far too small a sample to act on; a couple of unlucky/lucky calls swing that percentage wildly. Noise and not a segment recommendation.

On the other hand, **university.degree (14%)** outperform, while **basic.9y (8%) and basic.6y (8%)** — lower levels of schooling — underperform. Education correlates with job type, so this may be capturing overlapping signal with the job breakdown above rather than something fully independent.

In [0]:
SELECT segment_value AS marital_status, contacts, conversions, conversion_rate
FROM v_segment_conversion
WHERE segment_dimension = 'marital_status'
ORDER BY conversion_rate DESC;

marital_status,contacts,conversions,conversion_rate
unknown,80,12,0.15
single,11564,1620,0.1401
divorced,4611,476,0.1032
married,24921,2531,0.1016


### Observations
Marital status is a weak segment signal — **single (14%) vs. married (10%)** is a real but modest gap, and `unknown` (15%) is too small a sample (80 contacts) to trust. On its own, marital status probably isn't worth a targeting rule; it may be more useful as a secondary filter combined with age or job.

In [0]:
SELECT segment_value AS age_bracket, contacts, conversions, conversion_rate
FROM v_segment_conversion
WHERE segment_dimension = 'age_bracket'
ORDER BY conversion_rate DESC;

age_bracket,contacts,conversions,conversion_rate
66+,618,290,0.4693
null,5,2,0.4
18-25,1660,347,0.209
56-65,2963,451,0.1522
26-35,14844,1740,0.1172
46-55,8247,717,0.0869
36-45,12839,1092,0.0851


### Observations
Age shows the widest spread of any segment so far. **66+ converts at 47%** — more than 4x baseline — and even at 618 contacts that's a real, sizeable pattern, not noise. **18-25 (21%)** is also well above baseline. The middle of the working-age range, **36-45 (9%) and 46-55 (9)**, is where most of the volume sits (21,086 combined contacts), and conversion is well below baseline. This lines up with the job/education findings — retirees and students cluster at the age extremes, so age, job, and education are likely telling overlapping parts of the same story rather than four independent signals.

## Summary so far
Across all four segment cuts, the same shape repeats: students, retirees, and the 66+ / 18-25 age extremes convert far above baseline, while blue-collar/services workers and the 36-55 core working-age range — which make up the bulk of call volume — convert below baseline. These aren't four independent findings; they're overlapping views of what looks like one underlying pattern (life-stage / employment status), which we should keep in mind before turning this into a targeting rule.



### Call Efficiency 

Analyzing how many contact attempts does it take to land one conversion, and does that number vary by segment or by how the contact was made.

In [0]:
-- Does contanct method matter?
SELECT 
c.contact_type,
COUNT(*) as total_calls,
SUM(CAST(f.converted AS INT)) as total_conversions,
ROUND(AVG(CAST (f.converted AS INT)),4) AS overall_conversion_rate
FROM fact_contacts f
INNER JOIN dim_campaign c
ON c.campaign_key = f.campaign_key
GROUP BY c.contact_type;


contact_type,total_calls,total_conversions,overall_conversion_rate
telephone,15041,787,0.0523
cellular,26135,3852,0.1474


In [0]:
SELECT
  c.contact_type,
  COUNT(*) AS total_calls,
  ROUND(AVG(e.euribor3m), 4) AS avg_euribor3m,
  ROUND(MIN(e.euribor3m), 4) AS min_euribor3m,
  ROUND(MAX(e.euribor3m), 4) AS max_euribor3m
FROM fact_contacts f
INNER JOIN dim_campaign c ON f.campaign_key = c.campaign_key
INNER JOIN dim_economic_context e ON f.economic_context_key = e.economic_context_key
GROUP BY c.contact_type;

contact_type,total_calls,avg_euribor3m,min_euribor3m,max_euribor3m
telephone,15041,4.5356,0.634,5.045
cellular,26135,3.0951,0.634,4.97


### Observations
I checked whether contact_type (cellular vs telephone) affects conversion, and the raw numbers look huge — 5% for telephone vs 15% for cellular. But before trusting that, I checked if the two channels were used under different economic conditions, since we don't have real dates here.

Telephone calls average a much higher Euribor rate (4.5) than cellular calls (3.1), meaning telephone calls were used more during the earlier/higher-rate part of the campaign and cellular more during the later/lower-rate part. So this conversion gap is confounded with time/economic era — I can't tell yet if cellular itself converts better, or if conversion was just better later in the campaign for other reasons. Parking this until I do the economic-conditions analysis properly, and not using contact_type as a targeting recommendation on its own for now.

In [0]:
-- Does prior campaign history matter?
SELECT
  c.previous_outcome,
  COUNT(*) AS total_calls,
  SUM(CAST(f.converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS overall_conversion_rate
FROM fact_contacts f
INNER JOIN dim_campaign c
  ON c.campaign_key = f.campaign_key
GROUP BY c.previous_outcome
ORDER BY total_calls DESC;



previous_outcome,total_calls,total_conversions,overall_conversion_rate
nonexistent,35551,3140,0.0883
failure,4252,605,0.1423
success,1373,894,0.6511


### Observations

previous_outcome was a strong signal for conversion rate. Success (65.1%) is way above the overall baseline (11.3%), and failure (14.2%) is still slightly above baseline too — even though these are people who already said no once before. Prior contact of any kind, success or failure, beats no prior contact at all (nonexistent sits at just 8.8%). 

Someone the bank has already talked to before is generally a warmer lead than a total stranger, regardless of how that first conversation ended. Just because someone didn't convert last time doesn't seem to be a reason to stop recalling and re-offering them the product — the data suggests the opposite, that re-contacting past leads (successful or not) converts better than cold-contacting someone new.

In [0]:
-- Does it take more touches to convert or fewer?
SELECT
  f.converted,
  COUNT(*) AS total_calls,
  ROUND(AVG(f.campaign_number), 4) AS avg_touches_within_campaign
FROM fact_contacts f
GROUP BY f.converted;


converted,total_calls,avg_touches_within_campaign
false,36537,2.6334
true,4639,2.052


In [0]:
-- Conversion rate by number of touches within the current campaign
SELECT
  f.campaign_number AS touches_within_campaign,
  COUNT(*) AS total_calls,
  SUM(CAST(f.converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS conversion_rate
FROM fact_contacts f
GROUP BY f.campaign_number
ORDER BY touches_within_campaign;

touches_within_campaign,total_calls,total_conversions,conversion_rate
1,17634,2299,0.1304
2,10568,1211,0.1146
3,5340,574,0.1075
4,2650,249,0.094
5,1599,120,0.075
6,979,75,0.0766
7,629,38,0.0604
8,400,17,0.0425
9,283,17,0.0601
10,225,12,0.0533


### Observations
Two things point the same direction. First, converted rows average 2.1 touches within the campaign, while non-converted rows average 2.6 - so people who eventually said yes tended to say it earlier, not later. 

Second, there's a clear decline 13% on the first touch, down to 11.5% on the second touch, 10.8% on the third touch, and it keeps dropping -- by touch 5 it's roughly half of the first-touch rate (7.5%). 

Overall this indicates that most of the conversion opportunity is in the first few touches, and there's a real case for capping outreach around 4-5 attemps per person rather than continuing to redial well past that point - the marginal return drops off fast.

In [0]:
SELECT
  CASE
    WHEN f.campaign_number = 1 THEN '1'
    WHEN f.campaign_number BETWEEN 2 AND 3 THEN '2-3'
    WHEN f.campaign_number BETWEEN 4 AND 5 THEN '4-5'
    WHEN f.campaign_number BETWEEN 6 AND 7 THEN '6-7'
    ELSE '8+'
  END AS touch_bin,
  COUNT(*) AS total_calls,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS conversion_rate
FROM fact_contacts f
GROUP BY touch_bin;

touch_bin,total_calls,conversion_rate
1,17634,0.1304
2-3,15908,0.1122
4-5,4249,0.0868
6-7,1608,0.0703
8+,1777,0.0411


In [0]:
WITH capped AS (
  SELECT
    COUNT(*) AS calls_4plus,
    SUM(CAST(converted AS INT)) AS conversions_4plus
  FROM fact_contacts
  WHERE campaign_number >= 4
),
overall AS (
  SELECT
    COUNT(*) AS calls_total,
    SUM(CAST(converted AS INT)) AS conversions_total
  FROM fact_contacts
)
SELECT
  capped.calls_4plus,
  overall.calls_total,
  ROUND(capped.calls_4plus / overall.calls_total, 4) AS pct_of_calls_4plus,
  capped.conversions_4plus,
  overall.conversions_total,
  ROUND(capped.conversions_4plus / overall.conversions_total, 4) AS pct_of_conversions_4plus
FROM capped, overall;

calls_4plus,calls_total,pct_of_calls_4plus,conversions_4plus,conversions_total,pct_of_conversions_4plus
7634,41176,0.1854,555,4639,0.1196


### What this means
Capping outreach at 3 touches per lead eliminates 18.5% of total call volume (7,634 calls) while only giving up 12% of total conversions (555 conversions) — a favorable trade since the calls being cut convert well below baseline anyway.

To put a dollar figure on it, I'm using $5 per call, a rounded midpoint from a commonly cited industry range of $2.70-$5.60 per call (source: https://voiso.com/articles/call-center-cost-per-call/). This isn't from the dataset itself; it's an external assumption I'm applying on top, and I'm stating it explicitly rather than presenting it as if it came from the data.

At $5/call, that's 7,634 x $5 = $38,170 in estimated call center cost savings. Against that, I'd be forgoing an estimated 555 conversions, which I'm not putting a dollar value on since there's no deposit-value data in this dataset to monetize it with — leaving it as a plain count.

So the recommendation: cap outreach at 3 touches per lead. It saves a meaningful chunk of call volume/cost while only sacrificing a proportionally smaller share of conversions.

### Baseline Average Touches during campaign

In [0]:
SELECT
  COUNT(*) AS total_calls,
  ROUND(AVG(campaign_number), 4) AS overall_avg_touches
FROM fact_contacts;

total_calls,overall_avg_touches
41176,2.5679


### Observations

Overall average touches across all calls is 2.57. This is a baseline number, alongside the 11.3% conversion baseline. Overall average touches baseline will be used to determine whether a segment needs more or fewer touches than average to convert

In [0]:
CREATE OR REPLACE VIEW v_segment_conversion AS
WITH base AS (
  SELECT
    f.contact_key,
    f.converted,
    f.campaign_number, 
    c.job,
    c.education,
    c.marital_status,
    c.age_bracket
  FROM fact_contacts f
  JOIN dim_customer_profile c ON f.customer_profile_key = c.customer_profile_key
)
SELECT 'job' AS segment_dimension, job AS segment_value,
       COUNT(*) AS contacts,
       SUM(CAST(converted AS INT)) AS conversions,
       ROUND(AVG(CAST(converted AS INT)), 4) AS conversion_rate,
       ROUND(AVG(campaign_number), 4) AS avg_touches
FROM base GROUP BY job

UNION ALL

SELECT 'education', education,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4),
       ROUND(AVG(campaign_number), 4)
FROM base GROUP BY education

UNION ALL

SELECT 'marital_status', marital_status,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4),
       ROUND(AVG(campaign_number), 4)
FROM base GROUP BY marital_status

UNION ALL

SELECT 'age_bracket', age_bracket,
       COUNT(*), SUM(CAST(converted AS INT)),
       ROUND(AVG(CAST(converted AS INT)), 4),
       ROUND(AVG(campaign_number), 4)
FROM base GROUP BY age_bracket;

In [0]:
SELECT segment_value AS job, contacts, conversions, conversion_rate, avg_touches
FROM v_segment_conversion
WHERE segment_dimension = 'job'
ORDER BY conversion_rate DESC;

job,contacts,conversions,conversion_rate,avg_touches
student,875,275,0.3143,2.104
retired,1718,434,0.2526,2.4785
unemployed,1014,144,0.142,2.5641
admin.,10419,1351,0.1297,2.6236
management,2924,328,0.1122,2.4761
unknown,330,37,0.1121,2.6485
technician,6739,730,0.1083,2.5778
self-employed,1421,149,0.1049,2.6608
housemaid,1060,106,0.1,2.6396
entrepreneur,1456,124,0.0852,2.5357


In [0]:
SELECT segment_value AS education, contacts, conversions, conversion_rate, avg_touches
FROM v_segment_conversion
WHERE segment_dimension = 'education'
ORDER BY conversion_rate DESC;

education,contacts,conversions,conversion_rate,avg_touches
illiterate,18,4,0.2222,2.2778
unknown,1730,251,0.1451,2.5971
university.degree,12164,1669,0.1372,2.5637
professional.course,5240,595,0.1135,2.5868
high.school,9512,1031,0.1084,2.5689
basic.4y,4176,428,0.1025,2.6006
basic.6y,2291,188,0.0821,2.557
basic.9y,6045,473,0.0782,2.5323


In [0]:
SELECT segment_value AS marital_status, contacts, conversions, conversion_rate, avg_touches
FROM v_segment_conversion
WHERE segment_dimension = 'marital_status'
ORDER BY conversion_rate DESC;

marital_status,contacts,conversions,conversion_rate,avg_touches
unknown,80,12,0.15,3.1875
single,11564,1620,0.1401,2.5342
divorced,4611,476,0.1032,2.6133
married,24921,2531,0.1016,2.5731


In [0]:
SELECT segment_value AS age_bracket, contacts, conversions, conversion_rate, avg_touches
FROM v_segment_conversion
WHERE segment_dimension = 'age_bracket'
ORDER BY conversion_rate DESC;

age_bracket,contacts,conversions,conversion_rate,avg_touches
66+,618,290,0.4693,2.0113
null,5,2,0.4,2.2
18-25,1660,347,0.209,2.5337
56-65,2963,451,0.1522,2.6267
26-35,14844,1740,0.1172,2.5193
46-55,8247,717,0.0869,2.6532
36-45,12839,1092,0.0851,2.587


### Observations
Most segments sit close to the 2.57 overall average touches, so who you're calling doesn't change much about how many attempts it takes. That actually backs up the 3-touch cap finding: it's not something that only applies to certain segments, it holds pretty much across the board. The two exceptions are student and 66+, which have noticeably fewer touches than everyone else — and they're also the two best-converting segments, so they're the clearest case for prioritizing.

## EDA Summary
Overall baseline conversion is 11.3%. Job, education, and age_bracket all point at roughly the same pattern: students, retirees, and the 18-25/66+ age extremes convert well above baseline, while the core working-age, blue-collar/services crowd — most of the call volume — converts below it. Marital status is a weak signal on its own. previous_outcome turned out to be the strongest signal: prior contact of any kind (success or failure) beats no prior contact at all, and a prior success predicts a 65% conversion rate this time.

On effort: touches within a campaign show a clear diminishing-returns pattern, and capping outreach at 3 touches would cut 18.5% of call volume while only giving up 12% of conversions — about $38,000 in estimated savings at $5/call. That efficiency finding holds pretty evenly across segments, except for Students and those aged 66+ need noticeably fewer touches too, reinforcing them as the clearest priority group.

One open thread: contact_type (cellular vs telephone) showed a big raw conversion gap, but it turned out to be confounded with economic era, so I'm not using it as a targeting recommendation as-is. Next: checking economic conditions directly.

#### Does the economy at time of contact affect conversion?

In [0]:
-- Distribution of euribor3m

SELECT
    MIN(euribor3m) AS min_euribor3m,
    percentile(euribor3m, 0.25) AS p25_euribor3m,
    percentile(euribor3m, 0.5) AS median_euribor3m,
    percentile(euribor3m, 0.75) AS p75_euribor3m,
    MAX(euribor3m) AS max_euribor3m
FROM fact_contacts f
INNER JOIN dim_economic_context e 
ON f.economic_context_key = e.economic_context_key;



min_euribor3m,p25_euribor3m,median_euribor3m,p75_euribor3m,max_euribor3m
0.634,1.344,4.857,4.961,5.045


In [0]:
SELECT
  CASE
    WHEN e.euribor3m <= 1.344 THEN 'Q1 (low rate)'
    WHEN e.euribor3m <= 4.857 THEN 'Q2'
    WHEN e.euribor3m <= 4.961 THEN 'Q3'
    ELSE 'Q4 (high rate)'
  END AS euribor_quartile,
  COUNT(*) AS total_calls,
  SUM(CAST(f.converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS conversion_rate
FROM fact_contacts f
INNER JOIN dim_economic_context e ON f.economic_context_key = e.economic_context_key
GROUP BY euribor_quartile
ORDER BY euribor_quartile;

euribor_quartile,total_calls,total_conversions,conversion_rate
Q1 (low rate),10540,2675,0.2538
Q2,11509,950,0.0825
Q3,9342,475,0.0508
Q4 (high rate),9785,539,0.0551


In [0]:
-- Distribution for emp_var_rate and cons_conf_idx
SELECT
    MIN(e.emp_var_rate) AS min_emp_var_rate,
    percentile(e.emp_var_rate, 0.25) AS p25_emp_var_rate,
    percentile(e.emp_var_rate, 0.5) AS median_emp_var_rate,
    percentile(e.emp_var_rate, 0.75) AS p75_emp_var_rate,
    MAX(e.emp_var_rate) AS max_emp_var_rate,

    MIN(e.cons_conf_idx) AS min_cons_conf_idx,
    percentile(e.cons_conf_idx, 0.25) AS p25_cons_conf_idx,
    percentile(e.cons_conf_idx, 0.5) AS median_cons_conf_idx,
    percentile(e.cons_conf_idx, 0.75) AS p75_cons_conf_idx,
    MAX(e.cons_conf_idx) AS max_cons_conf_idx
FROM fact_contacts f
INNER JOIN dim_economic_context e 
ON f.economic_context_key = e.economic_context_key;

min_emp_var_rate,p25_emp_var_rate,median_emp_var_rate,p75_emp_var_rate,max_emp_var_rate,min_cons_conf_idx,p25_cons_conf_idx,median_cons_conf_idx,p75_cons_conf_idx,max_cons_conf_idx
-3.4,-1.8,1.1,1.4,1.4,-50.8,-42.7,-41.8,-36.4,-26.9


In [0]:
SELECT
  CASE
    WHEN e.emp_var_rate < 0 THEN 'negative (employment declining)'
    ELSE 'positive (employment growing)'
  END AS emp_var_rate_bin,
  COUNT(*) AS total_calls,
  SUM(CAST(f.converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS conversion_rate
FROM fact_contacts f
INNER JOIN dim_economic_context e ON f.economic_context_key = e.economic_context_key
GROUP BY emp_var_rate_bin;

emp_var_rate_bin,total_calls,total_conversions,conversion_rate
positive (employment growing),23990,1106,0.0461
negative (employment declining),17186,3533,0.2056


### Observations
20.6% conversion when employment was declining vs 4.6% when it was growing, even bigger than the Euribor split. Conversion was way better when employment was declining.

In [0]:
SELECT
  CASE
    WHEN e.cons_conf_idx <= -42.7 THEN 'Q1 (least confident)'
    WHEN e.cons_conf_idx <= -41.8 THEN 'Q2'
    WHEN e.cons_conf_idx <= -36.4 THEN 'Q3'
    ELSE 'Q4 (most confident)'
  END AS cons_conf_quartile,
  COUNT(*) AS total_calls,
  SUM(CAST(f.converted AS INT)) AS total_conversions,
  ROUND(AVG(CAST(f.converted AS INT)), 4) AS conversion_rate
FROM fact_contacts f
INNER JOIN dim_economic_context e 
ON f.economic_context_key = e.economic_context_key
GROUP BY cons_conf_quartile;

cons_conf_quartile,total_calls,total_conversions,conversion_rate
Q1 (least confident),15555,1651,0.1061
Q3,9832,1221,0.1242
Q2,7989,378,0.0473
Q4 (most confident),7800,1389,0.1781


### Economic conditions impact on conversion summary
Checked three economic indicators against conversion, and two of them tell a pretty consistent story. Euribor: conversion was way higher (25.4%) when the rate was in the lowest bucket, and everything else sat much lower (5-8%). emp_var_rate: even bigger gap, 20.6% conversion when employment was declining vs 4.6% when it was growing. Both point in the same direction - conversion was much better during Economically weaker periods. That said, these two probably aren't fully independent findings, since low interest rates and declining employment tend to happen together (same crisis era), so this is more like one underlying story showing up twice, not two separate confirmations.

This also finally explains something I couldn't earlier - the cellular vs telephone gap I found and parked. Telephone calls happened more during the Higher-rate period and converted worse (5.23%), cellular calls happened more during the lower-rate period and converted better (14.74%). Same direction as the euribor/emp_var_rate finding, so I think the phone-type gap was mostly picking up this same economic-conditions effect, not really about the channel itself.

cons_conf_idx didn't fit as cleanly - conversion went 10.61% -> 4.73% -> 12.42% -> 17.81% across the four confidence buckets, which isn't a clean trend, so I'm not using this one to make any claims.

Overall takeaway: when the campaign runs seems to matter a lot, maybe as much as who gets called. Weaker economic conditions (low rates, declining employment) line up with much better conversion.

# Final Targeting Recommendation

Three things converge into one recommendation: prioritize by life-stage, prioritize by prior contact history, and cap effort per lead.

**Who to prioritize (life-stage):** job, education, and age_bracket all point at the same underlying group rather than three separate findings - students (31.4%), retirees (25.3%), and the 18-25/66+ age brackets (20.9%, 46.9%) convert far above the 11.3% baseline. The core working-age, blue-collar/services crowd - most of the actual call volume - converts below baseline.

**Who to prioritize (prior contact):** the strongest signal I found. previous_outcome success converts at 65%, failure at 14%, nonexistent (never contacted) at just 9%. Any prior contact beats a cold one, even a past "no."

**How much effort per lead:** conversions cluster in the first few touches and drop off steadily after. Capping outreach at 3 touches eliminates 18.5% of call volume while only giving up 12% of conversions - about $38,000 in estimated savings at $5/call. This held evenly across almost every segment, so it's a rule that applies bank-wide, not something to customize per segment.

**When to call (economic conditions):** conversion was way higher during economically weaker stretches - 25.4% in the lowest Euribor quartile vs 5-8% elsewhere, 20.6% during declining employment vs 4.6% during growth. This also explained the earlier cellular vs telephone gap I'd parked - it wasn't really about the channel, it was mostly this same economic-timing effect.

**What I didn't use:** duration (leakage, only known after the call), contact_type (confounded with the economic-timing effect above), cons_conf_idx (no clean trend).

**Bottom line:** prioritize by life-stage and prior contact history, cap outreach at 3 attempts per lead, and treat economic conditions as a campaign-timing decision rather than a per-customer targeting filter.